# MISPR Workshop — DFT & MD Workflows
### Stony Brook University · MolMD Group

This notebook restores the **MISPR + MDPropTools** environment and opens **JupyterLab**,
where you run three tutorials:

| Notebook | What it does |
|---|---|
| `esp.ipynb` | Partial atomic charges (DFT / ESP workflow) |
| `binding_energy.ipynb` | Binding energy: CO2 + propionitrile (DFT) |
| `md.ipynb` | Electrolyte MD: density, diffusivity, RDF |

**You need only a Google account.** No installs, no licenses, no payment.

> The tutorials replay pre-computed Gaussian/LAMMPS outputs (`run_fake_gaussian` /
> `run_fake_lammps`), so they run in seconds on a free CPU runtime.

---
### How to use
1. **Runtime -> Run all** (or press play on the cell below).
2. Wait ~2 minutes.
3. Click the **JupyterLab link** that appears. Work there.
4. Keep **this tab open** — closing it shuts down the server.


In [ ]:
#@title Set up the workshop environment (run me - ~2 min) { display-mode: "form" }

# ============================================================================
#  EDIT THESE THREE URLS -> your repo + your two GitHub Release download links.
# ============================================================================
GITHUB_REPO = "https://github.com/kuldeepsinhraj/mispr-workshop-colab"
ENV_URL     = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/mispr-env.tar.gz"
DATA_URL    = "https://github.com/kuldeepsinhraj/mispr-workshop-colab/releases/download/v1.0/data.zip"
# ============================================================================

import os, time, shutil, subprocess
def sh(c):
    subprocess.run(c, shell=True, check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

t0 = time.time()
CONDA, CAPSULE = "/opt/conda", "/root/capsule"

# 1/5 environment (same /opt/conda prefix -> no relocation)
print("[1/5] Downloading the environment (~1 GB)...", flush=True)
if not os.path.exists(CONDA + "/bin/python"):
    sh("wget -q -O /tmp/env.tar.gz '%s'" % ENV_URL)
    sh("tar -xzf /tmp/env.tar.gz -C /")          # restores /opt/conda AND /opt/src
    subprocess.run(CONDA + "/bin/conda-unpack", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    sh("rm -f /tmp/env.tar.gz")
print("      env python:",
      subprocess.run([CONDA+"/bin/python","--version"],capture_output=True,text=True).stdout.strip())

# 2/5 notebooks + reference data
print("[2/5] Fetching notebooks and reference data...", flush=True)
os.makedirs(CAPSULE, exist_ok=True)
if not os.path.exists(CAPSULE + "/code"):
    sh("git clone --depth 1 -q '%s' /tmp/repo" % GITHUB_REPO)
    sh("cp -r /tmp/repo/code %s/code" % CAPSULE)
if not os.path.exists(CAPSULE + "/data"):
    fname = DATA_URL.split("/")[-1]              # data.zip or data.tar.gz
    sh("wget -q -O /tmp/%s '%s'" % (fname, DATA_URL))
    shutil.unpack_archive("/tmp/"+fname, CAPSULE)   # -> /root/capsule/data
os.makedirs(CAPSULE + "/results", exist_ok=True)

required = [CAPSULE+"/data/esp/water/ESP", CAPSULE+"/data/be/co2/Optimization",
            CAPSULE+"/data/md/runs", CAPSULE+"/data/md/result/npt/log.lammps"]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise SystemExit("Missing reference data - tutorials cannot run:\n  " + "\n  ".join(missing))

# 3/5 local MongoDB (offline, from the packed env)
print("[3/5] Starting local MongoDB...", flush=True)
os.makedirs(CAPSULE + "/db", exist_ok=True)
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = CONDA + "/lib:" + env.get("LD_LIBRARY_PATH", "")
subprocess.run([CONDA+"/bin/mongod","--dbpath",CAPSULE+"/db","--bind_ip","127.0.0.1",
                "--port","27017","--logpath",CAPSULE+"/db/mongod.log","--fork"],
               env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    if subprocess.run("ss -ltn 2>/dev/null | grep -q 27017", shell=True).returncode == 0: break
    time.sleep(1)
else:
    raise SystemExit("MongoDB failed to start. See " + CAPSULE + "/db/mongod.log")
print("      mongod on 127.0.0.1:27017")

# 4/5 fireworks config (capsule's config/*.yaml already correct -> only env var)
FW = CAPSULE + "/code/config/FW_config.yaml"

# 5/5 JupyterLab
print("[5/5] Launching JupyterLab...", flush=True)
lab_env = os.environ.copy()
lab_env["PATH"] = CONDA + "/bin:" + lab_env["PATH"]
lab_env["LD_LIBRARY_PATH"] = CONDA + "/lib:" + lab_env.get("LD_LIBRARY_PATH","")
lab_env["FW_CONFIG_FILE"] = FW
subprocess.Popen([CONDA+"/bin/jupyter","lab","--ip=0.0.0.0","--port=8888","--no-browser",
    "--allow-root","--notebook-dir="+CAPSULE+"/code","--ServerApp.token=",
    "--ServerApp.password=","--ServerApp.allow_origin=*","--ServerApp.disable_check_xsrf=True"],
    env=lab_env, stdout=open("/tmp/lab.log","w"), stderr=subprocess.STDOUT)
for _ in range(60):
    if subprocess.run("ss -ltn 2>/dev/null | grep -q 8888", shell=True).returncode == 0: break
    time.sleep(1)
else:
    raise SystemExit("JupyterLab failed to start. Check /tmp/lab.log")

print("\nReady in %ds. Click the link below to open JupyterLab.\n" % int(time.time()-t0))
from google.colab import output
output.serve_kernel_port_as_window(8888, path="/lab/tree/esp.ipynb")


---
## In JupyterLab
Run the notebooks in order (each finishes in under a minute):
1. **`esp.ipynb`**  2. **`binding_energy.ipynb`**  3. **`md.ipynb`**

**Skip each notebook's first cell** (`!service mongodb start`) — MongoDB is already
running, started above. If you run it, it just errors harmlessly; carry on.

Output lands in `/root/capsule/results/`.


In [ ]:
#@title (Optional) Save your results to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/mispr_workshop"
!cp -r /root/capsule/results "/content/drive/MyDrive/mispr_workshop/"
print("Saved to Google Drive -> MyDrive/mispr_workshop/results")
